# Experiment 2 protocol audit

Verify both scales, paired semantics, per-layer control-probe gates, and controller invariants before starting a long run.

In [ ]:
from pathlib import Path
import sys
import torch

REPO = Path.cwd()
if not (REPO / 'experiment_2').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'experiment_2/src'))

from experiment_2.config import load_config
from experiment_2.model import GPT, GPTConfig, projected_modules

rows=[]
for scale in ['level0','level1']:
    path=REPO / f'experiment_2/configs/{scale}.yaml'
    cfg=load_config(path)
    torch.manual_seed(1)
    model=GPT(GPTConfig(**cfg['model']))
    rows.append({
        'scale':scale,
        'parameters':sum(p.numel() for p in model.parameters()),
        'projected_matrices':len(projected_modules(model)),
        'steps':cfg['training']['max_steps'],
        'tokens_per_step':cfg['training']['batch_size']*cfg['training']['grad_accum_steps']*cfg['model']['block_size'],
        'control_interval':cfg['controller']['control_interval'],
        'projection_interval':cfg['controller']['projection_interval'],
        'max_active_layers':cfg['controller']['max_active_layers'],
        'target_alpha':cfg['controller']['target_alpha'],
        'enter_tolerance':cfg['controller']['enter_tolerance'],
        'exit_tolerance':cfg['controller']['exit_tolerance'],
        'probe_batches':cfg['controller']['probe_batches'],
        'max_probe_loss_increase':cfg['controller']['max_probe_loss_increase'],
    })
    assert cfg['controller']['target_alpha']==cfg['wwpgd']['target_alpha']==2.0
    assert cfg['analysis']['weightwatcher_interval']==cfg['controller']['control_interval']
    assert cfg['controller']['exit_tolerance'] < cfg['controller']['enter_tolerance']
    assert cfg['controller']['probe_batches'] >= 1
    assert cfg['controller']['max_probe_loss_increase'] >= 0

import pandas as pd
pd.DataFrame(rows)


The adaptive arm is explicitly **reference-guided**. Window credit is shared across the active cohort and is an approximate control signal, not a causal per-layer attribution. The fixed independent training-control probe provides a direct local loss gate for each candidate layer correction.